# Linear Regression from Scratch with Gradient Descent

<div style="background-color:#1e293b;padding:15px;border-left:6px solid #38bdf8;color:#e2e8f0">

<b>Linear regression</b> predicts a continuous number (a target) from one or more input features. With a single feature, the model is a line: <code>y_pred = w * x + b</code>, where <code>w</code> (the slope) and <code>b</code> (the intercept) are the parameters the model learns.

<b>How does it learn?</b> We need a way to measure how wrong a prediction is. The standard choice is the squared error <code>(y_pred - y)^2</code> for each sample. Averaged across the whole dataset this gives the <b>cost</b> — a single number that gets smaller as the model fits better.

<b>Gradient descent</b> is the optimization algorithm that minimises that cost. Starting from an initial guess of zero for <code>w</code> and <code>b</code>, it repeatedly nudges them in the direction that lowers cost — a tiny step at a time, controlled by the <b>learning rate</b>. Over many iterations the parameters settle on values that produce a good fit. Gradient descent is the engine behind every neural network — understanding it here is foundational for all of deep learning.

In this notebook you will build a working linear-regression-with-gradient-descent class yourself, in plain NumPy, and watch it learn.

</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)   # reproducible random numbers

---
## Step 1: A Toy Dataset We Designed Ourselves

Before touching real data, we will build the algorithm on a tiny dataset where we already know the right answer. The trick: we generate `y` from a known line, add a bit of noise, and then pretend we don't know the line — our model has to discover it from the data.

In [ ]:
# True line: y = 2 * x + 5  (plus some noise)
TRUE_W = 2.0
TRUE_B = 5.0

X = np.linspace(0, 2, 50)                      # 50 evenly spaced x values from 0 to 2
noise = np.random.normal(0, 0.5, size=len(X))  # small random noise
y = TRUE_W * X + TRUE_B + noise                # the y values, with noise added

plt.figure(figsize=(7, 4))
plt.scatter(X, y, alpha=0.7, label='observed data')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Our toy dataset')
plt.legend()
plt.show()

We know the true slope is `2` and the true intercept is `5`. Our model does not. Its job is to look at the dots and recover something close to those values.

---
## Step 2: The Linear Regression Model

The model is a line:

$$\text{y\_pred} = w \cdot x + b$$

`w` is the **weight** (slope — how much `x` influences the prediction) and `b` is the **bias** (intercept — the baseline prediction when `x = 0`). These terms carry directly into neural networks, where models have many weights and biases.

Two numbers are all it takes to describe the whole linear regression model with one feature. Once they are set, prediction is just plugging in `x`.

Start with a bad initial guess and see what the predictions look like.

In [ ]:
w_init = 0.0
b_init = 0.0

y_pred_init = w_init * X + b_init    # predictions with the bad initial guess

plt.figure(figsize=(7, 4))
plt.scatter(X, y, alpha=0.7, label='observed data')
plt.plot(X, y_pred_init, color='red', label=f'initial guess: y = {w_init}x + {b_init}')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.title('Initial (bad) model')
plt.show()

Clearly wrong — the red line predicts zero for every point. To improve the model we need (a) a way to score how bad it is, and (b) a procedure to update `w` and `b` so the score gets better.

---
## Step 3: Measuring Error — Loss and Cost

Two related terms you will hear constantly:

- **Loss** — the error on a single sample. For regression we use the squared error: $(\text{y\_pred} - y)^2$.
- **Cost** — the loss averaged across the whole dataset. For our problem the cost is **Mean Squared Error (MSE)**:

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (\text{y\_pred}_i - y_i)^2$$

Squared (rather than absolute) error has two convenient properties: it penalises big mistakes more than small ones, and it produces a smooth function we can take derivatives of — which is what gradient descent needs.

In [ ]:
def compute_loss(y_pred, y_true):
    """Squared error for each sample — returns an array of the same length as the inputs."""
    return (y_pred - y_true) ** 2

def compute_cost(y_pred, y_true):
    """Mean Squared Error — a single number summarising how wrong the model is across all samples."""
    return compute_loss(y_pred, y_true).mean()

# Cost of our bad initial guess
initial_cost = compute_cost(y_pred_init, y)
print(f"Initial cost (MSE): {initial_cost:.2f}")

That number is the model's score. Lower is better. Gradient descent will drive it down.

---
## Step 4: Gradient Descent — Following the Slope Downhill

Imagine the cost as a surface above all possible `(w, b)` pairs. We start at some random spot on that surface and want to walk to the lowest point.

The algorithm has six steps:

1. **Initialise** — set `w = 0`, `b = 0` (we have to start somewhere; for linear regression, zero works fine)
2. **Predict** — compute predictions using the current `w` and `b`
3. **Score** — compute the cost (MSE) to measure how wrong the model is
4. **Differentiate** — compute the gradients: how much does the cost change if we nudge `w` or `b` slightly?
5. **Step** — update: `w = w - learning_rate * dw` and `b = b - learning_rate * db`
6. **Repeat** — go back to step 2; repeat for a fixed number of iterations (`n_epochs`); the cost should keep falling until it levels off — that is **convergence**

Each pass through steps 2–6 is one **epoch** — a single full sweep over all training samples.

---

**Gradient formulas (step 4)** — for MSE with `y_pred = w*x + b`:

```
dw = (2/n) * sum( x * (y_pred - y) )
db = (2/n) * sum( y_pred - y )
```

The factor of `2` comes from differentiating the squared error; in practice it gets absorbed into the learning rate and does not affect the result.

> **Note:** In textbooks these are written as `∂cost/∂w` and `∂cost/∂b` — the `∂` symbol (called "partial derivative") just means "how much does cost change when we nudge this one parameter." The learning rate is often written as `η` (eta) or `α` (alpha). You do not need to understand the calculus. The code formulas above are the complete recipe.

**In plain words:**
- `dw`: multiply the prediction error `(y_pred − y)` by `x` for each sample, sum across all samples, and scale. When `x` is large and the error is large, `w` gets a bigger correction.
- `db`: the average prediction error across all samples. Since `b` shifts every prediction by the same amount regardless of `x`, its correction is just the average error.

You do not need to derive these — but you do need to translate them into NumPy code, which is exactly what the next step asks.

---
## Step 5: Build the Model

Put everything together as a class. The class stores the learning rate and number of epochs, holds the current `w` and `b`, runs gradient descent in `fit`, and keeps a history of the cost so we can plot it later.

Three TODOs to fill in. None require new ideas — translate Step 4's formulas into NumPy.

In [ ]:
class LinearRegressionGD:
    """Linear regression trained via batch gradient descent. Single feature: y = w*x + b."""

    def __init__(self, learning_rate=0.01, n_epochs=100):
        self.learning_rate = learning_rate
        self.n_epochs = n_epochs
        self.w = 0.0                 # zero init works for linear regression; neural networks need random init to break symmetry
        self.b = 0.0
        self.cost_history = []       # cost at the end of each epoch

    def predict(self, X):
        """Return predictions y_pred = w*x + b for every value in X."""
        # TODO 1: replace `None` with the prediction formula
        return None

    def fit(self, X, y):
        """Run gradient descent to learn w and b from data."""
        n = len(X)

        for epoch in range(self.n_epochs):
            y_pred = self.predict(X)
            cost = compute_cost(y_pred, y)
            self.cost_history.append(cost)

            # TODO 2: compute the gradients of the cost with respect to w and b.
            # Hint: use gradient update formulas from Step 4
            dw = None
            db = None

            # TODO 3: update the parameters using the gradient descent rule:
            #   param <- param - learning_rate * gradient
            self.w = None
            self.b = None

        return self

<details><summary><b>Solution — click to expand</b></summary>

```python
def predict(self, X):
    return self.w * X + self.b

# Inside fit():
dw = (2 / n) * np.sum(X * (y_pred - y))
db = (2 / n) * np.sum(y_pred - y)

self.w = self.w - self.learning_rate * dw
self.b = self.b - self.learning_rate * db
```

</details>

---
## Step 6: Train the Model

Once the TODOs are filled in, training is one line. Then we inspect what the model learned and compare to the true values we set in Step 1.

In [ ]:
model = LinearRegressionGD(learning_rate=0.1, n_epochs=100)
model.fit(X, y)

print(f"Learned w: {model.w:.3f}    (true value: {TRUE_W})")
print(f"Learned b: {model.b:.3f}    (true value: {TRUE_B})")
print(f"Final cost: {model.cost_history[-1]:.3f}")

If the TODOs are correct, the learned `w` and `b` should be close to `2.0` and `5.0`. Not exactly — the noise in the data prevents a perfect recovery — but in the same neighbourhood.

Two plots make what happened visible: the cost over epochs, and the fitted line on top of the data.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(model.cost_history)
plt.xlabel('epoch')
plt.ylabel('cost (MSE)')
plt.title('Cost falling as gradient descent runs')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(X, y, alpha=0.7, label='observed data')
plt.plot(X, model.predict(X), color='green', linewidth=2,
         label=f'learned: y = {model.w:.2f}x + {model.b:.2f}')
plt.plot(X, TRUE_W * X + TRUE_B, '--', color='black', alpha=0.5,
         label=f'true: y = {TRUE_W}x + {TRUE_B}')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.title('Learned line vs true line')
plt.show()

---
## Step 7: The Learning Rate Matters

The learning rate `η` (the step size) is the single most important hyperparameter in gradient descent. Too small and training crawls; too large and the algorithm overshoots and may never settle — or blow up entirely. Let's see all three behaviours on our data.

In [ ]:
model_low  = LinearRegressionGD(learning_rate=0.005, n_epochs=100).fit(X, y)
model_good = LinearRegressionGD(learning_rate=0.1,   n_epochs=100).fit(X, y)
model_high = LinearRegressionGD(learning_rate=0.5,   n_epochs=100).fit(X, y)

print(f"low  lr=0.005 -> w={model_low.w:.3f},  b={model_low.b:.3f},  final cost={model_low.cost_history[-1]:.3f}")
print(f"good lr=0.1   -> w={model_good.w:.3f}, b={model_good.b:.3f}, final cost={model_good.cost_history[-1]:.3f}")
print(f"high lr=0.5   -> w={model_high.w:.3g}, b={model_high.b:.3g}, final cost={model_high.cost_history[-1]:.3g}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(model_low.cost_history,  label='lr=0.005 (too low)')
plt.plot(model_good.cost_history, label='lr=0.1 (good)')
plt.plot(model_high.cost_history, label='lr=0.5 (too high)')
plt.xlabel('epoch')
plt.ylabel('cost (MSE)')
plt.yscale('log')   # log scale so we can see all three curves together
plt.title('How the learning rate shapes training')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

>  **Note:** Look at the printed numbers: the too-high run (lr=0.5) produced `w` and `b` over a hundred million and a cost around 3e+16 — that is **divergence**. The steps overshot so badly that each one made things worse rather than better. The too-low run (lr=0.005) is the opposite problem: cost is still above 1.6 at epoch 100, clearly still moving. Given enough epochs it would eventually converge, just very slowly. The log-scale cost plot lets you see all three behaviours at once.

**TODO:** Try a few more learning rates yourself in the cell below — what is the smallest learning rate that still reaches close to the true `w=2, b=5` within 100 epochs? What is the largest learning rate that doesn't diverge?

In [ ]:
# YOUR TURN: experiment with learning rates
m = LinearRegressionGD(learning_rate=0.05, n_epochs=100).fit(X, y)
print(f"w={m.w:.3f}, b={m.b:.3f}, final cost={m.cost_history[-1]:.3f}")

---
## Step 8: Try it on Real Data

Our class works on data we designed. The honest test: does it work on data we didn't?

We will use the **Salary vs. Years of Experience** dataset — 30 rows, one feature (`YearsExperience`) and one target (`Salary`). It is small but real.

In [ ]:
df = pd.read_csv('salary_data.csv')
df.head()

In [ ]:
# Feature and target. We rescale salary to thousands of dollars to keep the numbers small —
# raw salaries are in the tens of thousands, which produces huge gradients and forces a tiny
# learning rate. Predicting in thousands is just a units choice — the fitted line is identical.
X_salary = df['YearsExperience'].values
y_salary = df['Salary'].values / 1000.0

plt.figure(figsize=(7, 4))
plt.scatter(X_salary, y_salary, alpha=0.8)
plt.xlabel('Years of experience')
plt.ylabel('Salary (thousands of $)')
plt.title('Salary vs Years of Experience')
plt.show()

In [ ]:
salary_model = LinearRegressionGD(learning_rate=0.01, n_epochs=1000).fit(X_salary, y_salary)

print(f"Learned w: {salary_model.w:.3f}   (salary increases by ~${salary_model.w*1000:.0f} per year of experience)")
print(f"Learned b: {salary_model.b:.3f}   (predicted starting salary: ~${salary_model.b*1000:.0f})")

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(X_salary, y_salary, alpha=0.8, label='real data')
plt.plot(X_salary, salary_model.predict(X_salary), color='green', linewidth=2,
         label=f'learned: y = {salary_model.w:.2f}x + {salary_model.b:.2f}')
plt.xlabel('Years of experience')
plt.ylabel('Salary (thousands of $)')
plt.legend()
plt.title('Linear regression on real data')
plt.show()

---
## Step 9: Sanity Check Against scikit-learn

> **Note:** In practice, for linear regression you would use `sklearn.linear_model.LinearRegression`, which is faster and more numerically robust. We implemented gradient descent from scratch to understand how it works under the hood — that knowledge transfers directly to neural networks, where a closed-form solution doesn't exist.

scikit-learn's `LinearRegression` solves the same problem with a different method (a closed-form math formula instead of gradient descent). If our implementation is correct, the two should land in the same place.

In [ ]:
from sklearn.linear_model import LinearRegression

# sklearn expects X as a 2D array of shape (n_samples, n_features), so reshape
sk_model = LinearRegression().fit(X_salary.reshape(-1, 1), y_salary)

print(f"our class:       w={salary_model.w:.4f}, b={salary_model.b:.4f}")
print(f"sklearn version: w={sk_model.coef_[0]:.4f}, b={sk_model.intercept_:.4f}")

The two answers are close but not identical — the slope differs by about 1% and the intercept by a bit more. With more training epochs or a more carefully tuned learning rate, gradient descent would converge closer to sklearn's exact answer. The gap is a useful reminder: gradient descent is an iterative approximation. Sklearn's formula computes the exact answer in one step; our loop approaches it gradually.

---
## What You Should Be Able to Do Now

- Explain what `w` (weight) and `b` (bias) represent in `y_pred = w*x + b`
- Distinguish loss (per-sample error) from cost (averaged over the dataset)
- Describe what gradient descent does in plain words: take small steps in the direction that lowers the cost
- Implement one full epoch of gradient descent in NumPy
- Recognise what a too-low, just-right, and too-high learning rate look like on a cost curve
- Compare a hand-written model's coefficients to scikit-learn's and explain why they match

This notebook covered single-feature linear regression. The next notebook extends this to multiple features using scikit-learn.